In [1]:
import subprocess
subprocess.run(['pip', 'install', 'ase', 'pymatgen', '--break-system-packages', '-q'])

CompletedProcess(args=['pip', 'install', 'ase', 'pymatgen', '--break-system-packages', '-q'], returncode=0)

In [2]:
import numpy as np
from collections import Counter
from ase.build import surface, make_supercell
from ase.io import write, read
from ase.visualize import view
from ase import Atom
from pymatgen.io.ase import AseAtomsAdaptor
from pymatgen.io.lammps.data import LammpsData
from pymatgen.core.surface import Slab
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
import os

os.makedirs("slabs", exist_ok=True)
alloy = read('bulk_structure/MgZn2_relaxed.cif')
print(f"Bulk: {len(alloy)} atoms, {alloy.get_chemical_formula()}")

Bulk: 12 atoms, Mg4Zn8


In [3]:
slab = surface(alloy, (1, 0, 1), 12, vacuum=8)
sc = make_supercell(slab, [[2,0,0],[0,2,0],[0,0,1]])

sym = np.array(sc.get_chemical_symbols())
mg, zn = sum(sym == 'Mg'), sum(sym == 'Zn')

ps = AseAtomsAdaptor().get_structure(sc)
slab_obj = Slab(ps.lattice, ps.species, ps.frac_coords,
    miller_index=(1,0,1), oriented_unit_cell=ps, shift=0,
    scale_factor=[[1,0,0],[0,1,0],[0,0,1]])

print(f"Atoms: {len(sc)}, Mg: {mg}, Zn: {zn}")
print(f"Stoichiometric: {'YES' if zn == 2*mg else 'NO'}")
print(f"Symmetric: {slab_obj.is_symmetric()}")

Atoms: 576, Mg: 192, Zn: 384
Stoichiometric: YES
Symmetric: False


In [4]:
z = sc.get_positions()[:, 2] #gets z coordinate of every atom in slab. get_positions([:, 2] takes just the z column from array
sym = np.array(sc.get_chemical_symbols()) #gets whether its mg or zn
order = np.argsort(z) #gives atom indices sorted by z coordinate so its in order. order[0] is the index of the atom closest to the bottom, order[-1] is the topmost.

layers = []
layer_idx = []
cur = [order[0]] #starts new layer group with the bottomost atom 
for i in range(1, len(order)):
    if z[order[i]] - z[order[i-1]] < 0.3: #loop goes through atoms in z order, if atom is within 0.3A of last atom they are in same layer
        cur.append(order[i]) #when we go to a new layer we save the layers composition and indices and move to another cur
    else:
        layers.append(Counter(sym[j] for j in cur)) #just like atom counter- Mg 4 Zn 4 for example
        layer_idx.append(list(cur)) #actually holds atom indices in each plane
        cur = [order[i]]
layers.append(Counter(sym[j] for j in cur))
layer_idx.append(list(cur))

z_min, z_max = z.min(), z.max()
n = len(layers)

print(f"{'Bot':>7} {'Bot Comp':>10} {'from_bot':>9}  |  {'Top':>7} {'Top Comp':>10} {'from_top':>9} {'Match':>6}")
print("-" * 80)
for i in range(min(10, n//2)): #comparsion loop- compares layer 0 to layer 47 (top layer) and does this for first 10 pairs. to see if they align
    bot, top = layers[i], layers[n-1-i]
    zm_b = np.mean([z[j] for j in layer_idx[i]]) #average z position for atoms in that layer
    zm_t = np.mean([z[j] for j in layer_idx[n-1-i]])
    
    comp_b = f"Mg{bot.get('Mg',0)}Zn{bot.get('Zn',0)}" if bot.get('Mg',0) and bot.get('Zn',0) else (f"Mg{bot.get('Mg',0)}" if bot.get('Mg',0) else f"Zn{bot.get('Zn',0)}")
    comp_t = f"Mg{top.get('Mg',0)}Zn{top.get('Zn',0)}" if top.get('Mg',0) and top.get('Zn',0) else (f"Mg{top.get('Mg',0)}" if top.get('Mg',0) else f"Zn{top.get('Zn',0)}")
    
    match = "YES" if bot == top else "NO <---"
    print(f"{i:>7} {comp_b:>10} {zm_b-z_min:>9.3f}  |  {n-1-i:>7} {comp_t:>10} {z_max-zm_t:>9.3f} {match:>6}")
    #gives distances from top layer and distances from bottom layer
    #for symmetric, a layer at distance 1.3A from bottom surface and a layer at 1.3A from top surface should have same composition and match

    Bot   Bot Comp  from_bot  |      Top   Top Comp  from_top  Match
--------------------------------------------------------------------------------
      0        Zn8     0.000  |       95        Zn4     0.000 NO <---
      1        Mg4     0.606  |       94        Zn8     0.330 NO <---
      2     Mg4Zn4     1.318  |       93        Mg4     0.936 NO <---
      3        Zn4     1.709  |       92     Mg4Zn4     1.648 NO <---
      4     Mg4Zn4     2.100  |       91        Zn4     2.039 NO <---
      5        Mg4     2.812  |       90     Mg4Zn4     2.430 NO <---
      6        Zn8     3.417  |       89        Mg4     3.142 NO <---
      7        Zn4     3.748  |       88        Zn8     3.748 NO <---
      8        Zn8     4.078  |       87        Zn4     4.078 NO <---
      9        Mg4     4.684  |       86        Zn8     4.409 NO <---


In [5]:
top_zn4_mask = z > z_max - 0.5
keep = np.where(~top_zn4_mask)[0]
trimmed = sc[keep]

ps_trim = AseAtomsAdaptor().get_structure(trimmed)
slab_trim = Slab(ps_trim.lattice, ps_trim.species, ps_trim.frac_coords,
    miller_index=(1,0,1), oriented_unit_cell=ps_trim, shift=0,
    scale_factor=[[1,0,0],[0,1,0],[0,0,1]])

sym_t = np.array(trimmed.get_chemical_symbols())
print(f"After removing top Zn4:")
print(f"Atoms: {len(trimmed)}, Mg: {sum(sym_t=='Mg')}, Zn: {sum(sym_t=='Zn')}")
print(f"Symmetric: {slab_trim.is_symmetric()}")

sga = SpacegroupAnalyzer(ps_trim, symprec=0.1)
print(f"Space group: {sga.get_space_group_symbol()}")

for op in sga.get_symmetry_operations():
    if op.rotation_matrix[2][2] < 0:
        inv_translation = op.translation_vector
        print(f"\nInversion operation found:")
        print(f"  Maps fractional coord f -> {inv_translation} - f")
        break

After removing top Zn4:
Atoms: 564, Mg: 192, Zn: 372
Symmetric: False
Space group: Cm


In [6]:
bot_zn8_mask = z < z_min + 0.5
keep_bot = np.where(~bot_zn8_mask)[0]
trimmed_bot = sc[keep_bot]

ps_trim_bot = AseAtomsAdaptor().get_structure(trimmed_bot)
slab_trim_bot = Slab(ps_trim_bot.lattice, ps_trim_bot.species, ps_trim_bot.frac_coords,
    miller_index=(1,0,1), oriented_unit_cell=ps_trim_bot, shift=0,
    scale_factor=[[1,0,0],[0,1,0],[0,0,1]])

sym_tb = np.array(trimmed_bot.get_chemical_symbols())
print(f"After removing bottom Zn8:")
print(f"Atoms: {len(trimmed_bot)}, Mg: {sum(sym_tb=='Mg')}, Zn: {sum(sym_tb=='Zn')}")
print(f"Symmetric: {slab_trim_bot.is_symmetric()}")

if slab_trim_bot.is_symmetric():
    sga = SpacegroupAnalyzer(ps_trim_bot, symprec=0.1)
    print(f"Space group: {sga.get_space_group_symbol()}")
    for op in sga.get_symmetry_operations():
        if op.rotation_matrix[2][2] < 0:
            inv_translation = op.translation_vector
            print(f"Inversion: f -> {inv_translation} - f")
            break

# Also try removing BOTH top Zn4 and bottom Zn8
both_mask = (z > z_max - 0.5) | (z < z_min + 0.5)
keep_both = np.where(~both_mask)[0]
trimmed_both = sc[keep_both]

ps_trim_both = AseAtomsAdaptor().get_structure(trimmed_both)
slab_trim_both = Slab(ps_trim_both.lattice, ps_trim_both.species, ps_trim_both.frac_coords,
    miller_index=(1,0,1), oriented_unit_cell=ps_trim_both, shift=0,
    scale_factor=[[1,0,0],[0,1,0],[0,0,1]])

sym_tb2 = np.array(trimmed_both.get_chemical_symbols())
print(f"\nAfter removing both top Zn4 AND bottom Zn8:")
print(f"Atoms: {len(trimmed_both)}, Mg: {sum(sym_tb2=='Mg')}, Zn: {sum(sym_tb2=='Zn')}")
print(f"Symmetric: {slab_trim_both.is_symmetric()}")

if slab_trim_both.is_symmetric():
    sga2 = SpacegroupAnalyzer(ps_trim_both, symprec=0.1)
    print(f"Space group: {sga2.get_space_group_symbol()}")
    for op in sga2.get_symmetry_operations():
        if op.rotation_matrix[2][2] < 0:
            inv_translation = op.translation_vector
            print(f"Inversion: f -> {inv_translation} - f")
            break

# Try removing top 2 layers (Zn4 + Zn8)
top2_mask = z > z_max - 0.8
keep_top2 = np.where(~top2_mask)[0]
trimmed_top2 = sc[keep_top2]

ps_trim_top2 = AseAtomsAdaptor().get_structure(trimmed_top2)
slab_trim_top2 = Slab(ps_trim_top2.lattice, ps_trim_top2.species, ps_trim_top2.frac_coords,
    miller_index=(1,0,1), oriented_unit_cell=ps_trim_top2, shift=0,
    scale_factor=[[1,0,0],[0,1,0],[0,0,1]])

sym_t2 = np.array(trimmed_top2.get_chemical_symbols())
print(f"\nAfter removing top 2 layers (Zn4 + Zn8):")
print(f"Atoms: {len(trimmed_top2)}, Mg: {sum(sym_t2=='Mg')}, Zn: {sum(sym_t2=='Zn')}")
print(f"Symmetric: {slab_trim_top2.is_symmetric()}")

if slab_trim_top2.is_symmetric():
    sga3 = SpacegroupAnalyzer(ps_trim_top2, symprec=0.1)
    print(f"Space group: {sga3.get_space_group_symbol()}")
    for op in sga3.get_symmetry_operations():
        if op.rotation_matrix[2][2] < 0:
            inv_translation = op.translation_vector
            print(f"Inversion: f -> {inv_translation} - f")
            break

After removing bottom Zn8:
Atoms: 568, Mg: 192, Zn: 376
Symmetric: False

After removing both top Zn4 AND bottom Zn8:
Atoms: 556, Mg: 192, Zn: 364
Symmetric: True
Space group: C2/m
Inversion: f -> [0.32692308 0.33391304 0.99488491] - f

After removing top 2 layers (Zn4 + Zn8):
Atoms: 564, Mg: 192, Zn: 372
Symmetric: False


In [7]:
inv_translation = np.array([0.32692308, 0.33391304, 0.99488491])

# The 4 top Zn and 8 bottom Zn that were removed
# Get their positions from the original sc
z_sc = sc.get_positions()[:, 2]
sym_sc = np.array(sc.get_chemical_symbols())

top_zn4 = [i for i in range(len(sc)) if sym_sc[i] == 'Zn' and z_sc[i] > z_max - 0.5]
bot_zn8 = [i for i in range(len(sc)) if sym_sc[i] == 'Zn' and z_sc[i] < z_min + 0.5]

print(f"Top Zn4 ({len(top_zn4)} atoms):")
for i in top_zn4:
    pos = sc.positions[i]
    print(f"  idx={i}: ({pos[0]:.3f}, {pos[1]:.3f}, {pos[2]:.3f})")

print(f"\nBottom Zn8 ({len(bot_zn8)} atoms):")
for i in bot_zn8:
    pos = sc.positions[i]
    print(f"  idx={i}: ({pos[0]:.3f}, {pos[1]:.3f}, {pos[2]:.3f})")

# Calculate inversion partners for the top Zn4
ps_sc = AseAtomsAdaptor().get_structure(sc)
print(f"\nInversion partners of top Zn4 (should land near bottom):")
for i in top_zn4:
    frac = ps_sc[i].frac_coords
    inv_frac = inv_translation - frac
    inv_frac = inv_frac % 1.0
    inv_cart = ps_sc.lattice.get_cartesian_coords(inv_frac)
    print(f"  idx={i}: inv -> ({inv_cart[0]:.3f}, {inv_cart[1]:.3f}, {inv_cart[2]:.3f})")

print(f"\nInversion partners of bottom Zn8 (should land near top):")
for i in bot_zn8:
    frac = ps_sc[i].frac_coords
    inv_frac = inv_translation - frac
    inv_frac = inv_frac % 1.0
    inv_cart = ps_sc.lattice.get_cartesian_coords(inv_frac)
    print(f"  idx={i}: inv -> ({inv_cart[0]:.3f}, {inv_cart[1]:.3f}, {inv_cart[2]:.3f})")

Top Zn4 (12 atoms):
  idx=137: (7.583, 3.456, 56.607)
  idx=141: (9.430, 1.553, 56.277)
  idx=142: (10.139, 4.163, 56.277)
  idx=281: (8.980, 8.605, 56.607)
  idx=285: (10.828, 6.702, 56.277)
  idx=286: (11.536, 9.312, 56.277)
  idx=425: (17.768, 3.456, 56.607)
  idx=429: (19.616, 1.553, 56.277)
  idx=430: (20.324, 4.163, 56.277)
  idx=569: (19.165, 8.605, 56.607)
  idx=573: (21.013, 6.702, 56.277)
  idx=574: (21.722, 9.312, 56.277)

Bottom Zn8 (8 atoms):
  idx=6: (8.348, 1.885, 8.000)
  idx=7: (9.037, 4.424, 8.000)
  idx=150: (9.746, 7.034, 8.000)
  idx=151: (10.434, 9.573, 8.000)
  idx=294: (18.534, 1.885, 8.000)
  idx=295: (19.223, 4.424, 8.000)
  idx=438: (19.931, 7.034, 8.000)
  idx=439: (20.620, 9.573, 8.000)

Inversion partners of top Zn4 (should land near bottom):
  idx=137: inv -> (2.805, 10.280, 7.670)
  idx=141: inv -> (18.534, 1.885, 8.000)
  idx=142: inv -> (20.620, 9.573, 8.000)
  idx=281: inv -> (1.408, 5.131, 7.670)
  idx=285: inv -> (19.931, 7.034, 8.000)
  idx=286: in

In [9]:
unpaired = [137, 281, 425, 569]

print("Unpaired top Zn and their inversion partners:")
for idx in unpaired:
    cart = sc.positions[idx]
    frac = ps_sc[idx].frac_coords
    inv_frac = (inv_translation - frac) % 1.0
    inv_cart = ps_sc.lattice.get_cartesian_coords(inv_frac)
    print(f"  idx={idx}: ({cart[0]:.3f}, {cart[1]:.3f}, {cart[2]:.3f}) "
          f"-> inv: ({inv_cart[0]:.3f}, {inv_cart[1]:.3f}, {inv_cart[2]:.3f})")

# Keep 425, 569 on top. Move 137 -> inv(425), move 281 -> inv(569)
sc_fixed = sc.copy()

frac_425 = ps_sc[425].frac_coords
cart_inv_425 = ps_sc.lattice.get_cartesian_coords((inv_translation - frac_425) % 1.0)

frac_569 = ps_sc[569].frac_coords
cart_inv_569 = ps_sc.lattice.get_cartesian_coords((inv_translation - frac_569) % 1.0)

sc_fixed.positions[137] = cart_inv_425
sc_fixed.positions[281] = cart_inv_569

print(f"\nMoved 137 -> inv(425): ({cart_inv_425[0]:.3f}, {cart_inv_425[1]:.3f}, {cart_inv_425[2]:.3f})")
print(f"Moved 281 -> inv(569): ({cart_inv_569[0]:.3f}, {cart_inv_569[1]:.3f}, {cart_inv_569[2]:.3f})")

sym_f = np.array(sc_fixed.get_chemical_symbols())
mg = sum(sym_f == 'Mg')
zn = sum(sym_f == 'Zn')

ps_fixed = AseAtomsAdaptor().get_structure(sc_fixed)
slab_fixed = Slab(ps_fixed.lattice, ps_fixed.species, ps_fixed.frac_coords,
    miller_index=(1,0,1), oriented_unit_cell=ps_fixed, shift=0,
    scale_factor=[[1,0,0],[0,1,0],[0,0,1]])

print(f"\nAtoms: {len(sc_fixed)}, Mg: {mg}, Zn: {zn}")
print(f"Stoichiometric: {'YES' if zn == 2*mg else 'NO'}")
print(f"Symmetric: {slab_fixed.is_symmetric()}")

Unpaired top Zn and their inversion partners:
  idx=137: (7.583, 3.456, 56.607) -> inv: (2.805, 10.280, 7.670)
  idx=281: (8.980, 8.605, 56.607) -> inv: (1.408, 5.131, 7.670)
  idx=425: (17.768, 3.456, 56.607) -> inv: (12.990, 10.280, 7.670)
  idx=569: (19.165, 8.605, 56.607) -> inv: (11.593, 5.131, 7.670)

Moved 137 -> inv(425): (12.990, 10.280, 7.670)
Moved 281 -> inv(569): (11.593, 5.131, 7.670)

Atoms: 576, Mg: 192, Zn: 384
Stoichiometric: YES
Symmetric: True


In [10]:
view(sc_fixed)

<Popen: returncode: None args: ['C:\\Users\\Kai Savage\\anaconda3\\python.ex...>

In [11]:
thick = sc_fixed.get_positions()[:,2].max() - sc_fixed.get_positions()[:,2].min()
area = np.linalg.norm(np.cross(sc_fixed.cell[0], sc_fixed.cell[1]))

ps = AseAtomsAdaptor().get_structure(sc_fixed)
ld = LammpsData.from_structure(ps, atom_style="atomic")
ld.write_file("slabs/mgzn2_101_12layers_reconstructed_ase.data")

print(f"12-layer (10-11) slab saved")
print(f"  File: slabs/mgzn2_101_12layers_reconstructed_ase.data")
print(f"  Atoms: {len(sc_fixed)}")
print(f"  Thickness: {thick:.1f} A")
print(f"  Area: {area:.1f} A²")
print(f"  Stoichiometric: YES")
print(f"  Symmetric: TRUE")

12-layer (10-11) slab saved
  File: slabs/mgzn2_101_12layers_reconstructed_ase.data
  Atoms: 576
  Thickness: 48.9 A
  Area: 209.8 A²
  Stoichiometric: YES
  Symmetric: TRUE


In [12]:
#unrelaxed surface energy calculation
E_slab = -759.47789         # eV
E_bulk_per_fu = -16.195843 / 4   # eV per formula unit = -4.048961 eV
n = 576 / 3                 # formula units in slab = 32
A = 209.8              # Ang^2

S_eV_per_ang2 = (E_slab - n * E_bulk_per_fu) / (2 * A)

# Convert to J/m^2 (1 eV/Ang^2 = 16.0218 J/m^2)
S_J_per_m2 = S_eV_per_ang2 * 16.0218

print(f"E_bulk per formula unit = {E_bulk_per_fu:.6f} eV")
print(f"n * E_bulk = {n * E_bulk_per_fu:.5f} eV")
print(f"E_slab - n*E_bulk = {E_slab - n * E_bulk_per_fu:.5f} eV")
print(f"S = {S_eV_per_ang2:.6f} eV/Ang^2")
print(f"S = {S_J_per_m2:.4f} J/m^2")

E_bulk per formula unit = -4.048961 eV
n * E_bulk = -777.40046 eV
E_slab - n*E_bulk = 17.92257 eV
S = 0.042713 eV/Ang^2
S = 0.6843 J/m^2


In [14]:
#relaxed surface energy calculation
E_slab_relaxed =  -760.28653209941  # eV
E_bulk_per_fu = -16.195843 / 4  # eV per formula unit
n = 576 / 3                      # 32 formula units
A = 209.8                 # Ang^2

S_relaxed_eV = (E_slab_relaxed - n * E_bulk_per_fu) / (2 * A)
S_relaxed_Jm2 = S_relaxed_eV * 16.0218

print(f"E_slab_relaxed - n*E_bulk = {E_slab_relaxed - n * E_bulk_per_fu:.5f} eV")
print(f"S relaxed = {S_relaxed_eV:.6f} eV/Ang^2")
print(f"S relaxed = {S_relaxed_Jm2:.4f} J/m^2")

# Compare with unrelaxed
S_unrelaxed_Jm2 = 0.6843
print(f"\nUnrelaxed S = {S_unrelaxed_Jm2:.4f} J/m^2")
print(f"Relaxed S   = {S_relaxed_Jm2:.4f} J/m^2")
print(f"Relaxation energy = {S_unrelaxed_Jm2 - S_relaxed_Jm2:.4f} J/m^2")

E_slab_relaxed - n*E_bulk = 17.11393 eV
S relaxed = 0.040786 eV/Ang^2
S relaxed = 0.6535 J/m^2

Unrelaxed S = 0.6843 J/m^2
Relaxed S   = 0.6535 J/m^2
Relaxation energy = 0.0308 J/m^2
